# Урок 4.3. Мониторинг RAG в продакшене: логи, метрики, алерты

Урок 4.1 закончился главным антипаттерном: RAG деградирует **тихо**. Урок 4.2
построил оборону — но её работу мы видели только глазами, в ноутбуке. В проде
никто не сидит и не смотрит в ноутбук. Нужно, чтобы сервис **сам рассказывал
о себе**: что происходит внутри, как быстро, и когда пора бить тревогу.

> «Систему, которую не видно, невозможно чинить. Её можно только перезапускать
> и надеяться.»

Сегодня оснащаем наш RAG-сервис тремя классическими инструментами
наблюдаемости (observability) и учимся отличать их друг от друга — потому что
их постоянно путают.

## 🎯 Цели урока
- Разобрать **три столпа** наблюдаемости: логи, метрики, трейсы — что, когда и
  зачем.
- Настроить структурные (JSON) логи с `trace_id` и понять, **что нельзя
  логировать**.
- Считать метрики Prometheus в FastAPI и понять модель **pull**, а не push.
- Увидеть, почему **среднее лжёт**, и why нужны перцентили (p95/p99).
- Сформулировать **алерты** — правила, которые превращают метрики в звонок.


## Три столпа: не синонимы

```
┌ ЛОГИ ────────────┬ МЕТРИКИ ───────────┬ ТРЕЙСЫ ──────────────┐
│ что произошло в   │ агрегаты по ВСЕМ    │ сколько заняла каждая │
│ ОДНОМ запросе     │ запросам во времени │ стадия ОДНОГО запроса │
│ детально, дорого  │ дёшево, для алертов │ где именно медленно   │
│ «упал на docid=X» │ «p95 latency 8с»    │ retrieve 20мс,        │
│                   │ «отказов 12%»       │ llm 7.5с ← вот тут    │
└───────────────────┴─────────────────────┴───────────────────────┘
           всё связано одним trace_id
```

Типичная ошибка — пытаться алертить по логам («грепаем ERROR») или хранить в
логах то, что должно быть метрикой. Правило: **метрики отвечают "плохо ли
сейчас?", логи — "что именно случилось в запросе X?", трейсы — "на какой
стадии тормозит?"**. Наш `rag_service/observability.py` даёт все три; разберём
по очереди.


In [ ]:
import os
import sys
from pathlib import Path

MODULE_DIR = Path.cwd()
if not (MODULE_DIR / "rag_service").exists() and (MODULE_DIR.parent / "rag_service").exists():
    MODULE_DIR = MODULE_DIR.parent
sys.path.insert(0, str(MODULE_DIR))

In [ ]:
from rag_service.store import KnowledgeBase
from rag_service.llm import OllamaLLM

kb = KnowledgeBase()
QDRANT_UP = kb.available() and kb.collection_exists()
OLLAMA_UP = OllamaLLM().available()
from rag_service.observability import PROMETHEUS_AVAILABLE
print(f"Qdrant: {'✓' if QDRANT_UP else '✗'} | Ollama: {'✓' if OLLAMA_UP else '✗'} | "
      f"prometheus_client: {'✓' if PROMETHEUS_AVAILABLE else '✗ (pip install prometheus-client)'}")

## Столп 1. Структурные логи

Главное правило продовых логов: **лог — это данные, а не предложение на
человеческом языке**. Не `log.info(f"ответил за {ms} мс на {question}")`, а
событие с полями: `{"event": "ask.done", "duration_ms": 1742, "outcome": "answered"}`.
Такую строку парсит Loki/ELK/Datadog без regex-магии; по ней можно строить
запросы «покажи все ask.done, где duration_ms > 5000».

`configure_logging` умеет два формата из одного кода: человекочитаемый для
разработки и JSON для прода. Сам код логирования (`log_event`) не меняется.

In [ ]:
from rag_service.observability import configure_logging, log_event

print("--- dev-формат (json_logs=False): читаем глазами ---")
configure_logging("INFO", json_logs=False, service="rag")
log_event("ask.done", outcome="answered", duration_ms=1742, n_sources=4)

print("\n--- prod-формат (json_logs=True): парсит машина ---")
configure_logging("INFO", json_logs=True, service="rag")
log_event("ask.done", outcome="answered", duration_ms=1742, n_sources=4)

Одно событие, два представления. В проде включаете JSON (`RAG_JSON_LOGS=1`
— это дефолт), локально — текст. Поля (`outcome`, `duration_ms`) идут в лог
структурно, а не заклеены в строку.

### trace_id: сшиваем строки одного запроса

Один запрос порождает десяток лог-строк из разных функций. Чтобы собрать их
вместе, каждой присваивается общий `trace_id`. Хранится он в `contextvars`
(request-scoped переменная) — не нужно протаскивать его параметром через
десять функций.

In [ ]:
from rag_service.observability import new_trace_id, get_trace_id

configure_logging("INFO", json_logs=False, service="rag")
tid = new_trace_id()
print("Новый trace_id запроса:", tid)
log_event("ask.start", question_len=30)
log_event("retrieve.done", hits=4)
log_event("ask.done", outcome="answered")
print("\nВсе три строки выше несут один trace_id =", get_trace_id())
print("По нему в Kibana/Loki собирается вся история запроса одним фильтром.")

### Что НЕЛЬЗЯ логировать

Логи оседают надолго и доступны широкому кругу. В них **не должно** быть:
- **ПДн** (телефоны, почты, карты) — вместо текста вопроса логируем его длину и
  хеш;
- **секретов** (ключи, токены, пароли);
- **полных промптов и ответов** — это и ПДн, и утечка содержимого документов.

`scrub_for_logs` маскирует ПДн и секреты и обрезает длину — прогоняйте через
него всё, что идёт в лог из пользовательского ввода или из документов.

In [ ]:
from rag_service.observability import log_event
from rag_service.security import scrub_for_logs
import hashlib

dirty = "Мой телефон +7 916 555-31-27, карта 4276 1600 5555 0918, помогите с VPN"
print("Как есть (ТАК НЕЛЬЗЯ):", dirty)
print("scrub_for_logs:        ", scrub_for_logs(dirty))

# Ещё безопаснее для приватности — логировать не текст, а его отпечаток:
qhash = hashlib.sha256(dirty.encode()).hexdigest()[:12]
print(f"\nВ проде часто логируют только: question_len={len(dirty)}, question_hash={qhash}")
print("Так можно сгруппировать одинаковые вопросы, не храня ни одного из них.")

## Столп 2. Метрики Prometheus

Метрика — это **число, меняющееся во времени**, которое дёшево агрегировать по
всем запросам. Три основных типа:

- **Counter** — только растёт (число запросов, отказов, отбитых инъекций).
- **Gauge** — прыгает вверх-вниз (число чанков в индексе, время последней
  индексации).
- **Histogram** — распределение значений по корзинам (латентность стадий,
  скор лучшего чанка) — из него считаются перцентили.

Наш сервис уже объявил их в `observability.py`. Ключевая опасность —
**cardinality explosion**: каждое уникальное значение метки (label) создаёт
отдельный временной ряд. Положите `user_id` или текст вопроса в метку — и
Prometheus задохнётся.

In [ ]:
from rag_service import observability as obs

print("Метрики, объявленные сервисом (имя — назначение):")
for name in dir(obs):
    if name.startswith("RAG_"):
        metric = getattr(obs, name)
        doc = getattr(metric, "_documentation", "") if hasattr(metric, "_documentation") else ""
        print(f"  {name:24s} {doc}")

print("\nПРАВИЛЬНЫЕ метки (мало значений):")
print("  rag_requests_total{outcome, tenant}  — outcome ∈ {answered, refused_no_context, ...}")
print("ОПАСНЫЕ метки (бесконечно значений — НЕ делайте так):")
print("  rag_requests_total{user_id, question, trace_id}  → cardinality explosion")

## Столп 3. Трейсинг стадий

Метрика говорит «p95 latency = 8 секунд». Трейс говорит **где** эти 8 секунд.
Контекст-менеджер `stage()` замеряет каждую стадию, пишет её в лог и в
гистограмму `rag_stage_duration_seconds{stage=...}`. Прогоним живой запрос и
посмотрим на разбивку по стадиям.

In [ ]:
from rag_service.pipeline import RagPipeline
from rag_service.access import DEMO_PRINCIPALS

if QDRANT_UP and OLLAMA_UP:
    configure_logging("INFO", json_logs=False, service="rag")
    pipe = RagPipeline(kb=kb, llm=OllamaLLM())
    pipe.build_lexical_index()
    print("Трейс одного запроса (каждая stage.done — стадия с длительностью):\n")
    res = pipe.ask("За сколько дней подавать заявку на отпуск?",
                   DEMO_PRINCIPALS["demo-employee-token"])
    print(f"\nИтог: {res.outcome}, llm_ms={round(res.llm.duration_s*1000,1) if res.llm else '-'}")
else:
    print("Нужны Qdrant + Ollama. Типичная разбивка (мой прогон):")
    print("  retrieve.dense ~16-20мс | retrieve.bm25 ~0.1мс | llm.generate ~1700-2500мс")
    print("Вывод очевиден и почти всегда одинаков: узкое место RAG — это LLM, не поиск.")


Разбивка почти всегда одна и та же: поиск — единицы-десятки миллисекунд,
LLM — секунды. **Узкое место RAG — генерация, а не retrieval.** Это знание
экономит часы: оптимизировать поиск, когда 99% времени уходит на LLM,
бессмысленно.

### Почему среднее лжёт: холодный старт

Замерим латентность LLM на нескольких запросах подряд — и увидим, почему по
метрикам смотрят перцентили, а не среднее.

In [ ]:
import time

if QDRANT_UP and OLLAMA_UP:
    configure_logging("WARNING", json_logs=False)  # приглушим логи, оставим замеры
    latencies = []
    for i in range(4):
        res = pipe.ask("Сколько дней отпуска в 2026?", DEMO_PRINCIPALS["demo-employee-token"])
        ms = res.llm.duration_s * 1000 if res.llm else 0
        latencies.append(ms)
        print(f"запрос {i+1}: llm={ms:7.0f} мс")
    mean = sum(latencies) / len(latencies)
    print(f"\nсреднее: {mean:.0f} мс | максимум: {max(latencies):.0f} мс")
    print("Если первый запрос был холодным (модель грузилась в память) — среднее")
    print("окажется где-то посередине и не опишет НИ один реальный запрос.")
else:
    print("Мой прогон в уроке: ПЕРВЫЙ вызов qwen занял ~37 секунд (модель грузилась")
    print("в VRAM — холодный старт), последующие — 1.5-2.5 секунды.")
    print("Среднее (~10с) не описывает никого: ни холодный запрос, ни тёплые.")
    print("Поэтому алертят по p95/p99, а холодный старт лечат keep_alive (модуль 5).")


В моём прогоне при написании урока **первый** вызов qwen занял ~37 секунд
(модель грузилась в VRAM), последующие — 1.5–2.5 секунды. Среднее (~10 с) не
описывает ни один реальный запрос: тёплые быстрее, холодный кратно медленнее.

Вот зачем **Histogram и перцентили**: `p50` (медиана) покажет типичный запрос,
`p95/p99` — «хвост», который и портит жизнь пользователям. Prometheus считает
их из корзин гистограммы запросом `histogram_quantile(0.95, ...)`. Холодный
старт как явление лечится на уровне Ollama (`keep_alive`, прогрев) — это модуль 5.


## Метрики в FastAPI: эндпоинт /metrics

Prometheus работает по модели **pull**: сервис не шлёт метрики никуда сам —
он лишь отдаёт их текстом на `GET /metrics`, а сервер Prometheus раз в
`scrape_interval` (обычно 15 с) приходит и забирает. Поднимем приложение через
`TestClient`, сделаем несколько запросов и снимем метрики — ровно как это
сделал бы Prometheus.

In [ ]:
from fastapi.testclient import TestClient
from rag_service.app import create_app

if QDRANT_UP and OLLAMA_UP:
    with TestClient(create_app()) as client:
        # немного трафика: и обычные вопросы, и заведомый отказ
        asks = [
            ("demo-employee-token", "Сколько дней отпуска в 2026?"),
            ("demo-employee-token", "За сколько подавать заявку на отпуск?"),
            ("demo-employee-token", "Какая вилка у сеньора бэкенда?"),  # нет прав → откажет
        ]
        for token, q in asks:
            r = client.post("/ask", json={"question": q}, headers={"X-API-Token": token})
            print(f"  {r.json()['outcome']:20s} trace={r.headers.get('X-Trace-Id')}  «{q[:35]}»")

        metrics_text = client.get("/metrics").text
    print("\n--- фрагмент /metrics (то, что заберёт Prometheus) ---")
    for line in metrics_text.splitlines():
        if line.startswith(("rag_requests_total", "rag_pii_masked_total")) and "{" in line:
            print(" ", line)
else:
    print("Нужны Qdrant + Ollama. /metrics отдаёт текст в формате Prometheus, например:")
    print("  rag_requests_total{outcome=\"answered\",tenant=\"vectorika\"} 2.0")
    print("  rag_requests_total{outcome=\"refused_no_answer\",tenant=\"vectorika\"} 1.0")

Это и есть весь «интерфейс» сервиса для Prometheus — простой текст. В проде
рядом ставят готовый `prometheus-fastapi-instrumentator`, который добавляет
стандартные HTTP-метрики (латентность, коды ответов) автоматически; наши
доменные метрики (`rag_requests_total`, `rag_injection_blocked_total`) — сверху,
их пишет сам пайплайн.


## Алерты: превращаем метрики в звонок

Метрики без алертов — это дашборд, на который никто не смотрит. Алерт — правило
вида «если метрика ушла за порог N минут — разбуди дежурного». В настоящем
Prometheus это правила на языке **PromQL**, например:

```promql
# p95 латентности ответа выше 10 секунд последние 5 минут
histogram_quantile(0.95, rate(rag_stage_duration_seconds_bucket{stage="llm.generate"}[5m])) > 10

# доля отказов «нет контекста» выше 30%
sum(rate(rag_requests_total{outcome="refused_no_context"}[10m]))
  / sum(rate(rag_requests_total[10m])) > 0.3

# всплеск отбитых инъекций — возможно, идёт атака
sum(rate(rag_injection_blocked_total[5m])) > 1
```

Смысл каждого — «плохо ли *сейчас*, на потоке». Чтобы пощупать логику без
сервера Prometheus, посчитаем долю отказов прямо из текста `/metrics`.

In [ ]:
import re

def read_counter(metrics_text, name, **labels):
    """Достаёт значение счётчика с нужными метками из текста /metrics."""
    total = 0.0
    for line in metrics_text.splitlines():
        if not line.startswith(name) or line.startswith("#"):
            continue
        if all(f'{k}="{v}"' in line for k, v in labels.items()):
            total += float(line.rsplit(" ", 1)[-1])
    return total

def check_alerts(metrics_text):
    """Простые пороговые правила по счётчикам (аналог PromQL, но в Python)."""
    total = read_counter(metrics_text, "rag_requests_total")
    alerts = []
    if total >= 1:
        refused = (read_counter(metrics_text, "rag_requests_total", outcome="refused_no_context")
                   + read_counter(metrics_text, "rag_requests_total", outcome="refused_no_answer"))
        blocked = read_counter(metrics_text, "rag_requests_total", outcome="blocked_output")
        if refused / total > 0.5:
            alerts.append(f"⚠️ доля отказов {refused/total:.0%} > 50% — проверьте индекс/порог")
        if blocked / total > 0.2:
            alerts.append(f"⚠️ доля заблокированных ответов {blocked/total:.0%} > 20% — возможна атака")
    return alerts or ["✓ всё в норме"]

if QDRANT_UP and OLLAMA_UP:
    print("Правила на текущих метриках сервиса:")
    for a in check_alerts(metrics_text):
        print(" ", a)
else:
    demo = ("rag_requests_total{outcome=\"answered\"} 2.0\n"
            "rag_requests_total{outcome=\"refused_no_answer\"} 3.0\n")
    print("Демо на синтетических метриках (3 отказа из 5):")
    for a in check_alerts(demo):
        print(" ", a)


**На что стоит алертить в RAG-сервисе:**

| Метрика | Порог-пример | О чём говорит |
|---|---|---|
| p95 `llm.generate` | > 10 с | модель тормозит / очередь / холодные старты |
| доля `refused_no_context` | > 30% | сломался поиск, устарел индекс, съехал порог |
| доля `blocked_output` | > 20% | атака или регрессия промпта |
| `rag_injection_blocked` rate | всплеск | идёт атака через документы/ввод |
| `rag_index_last_success` | > 24 ч назад | переиндексация встала — данные протухают |
| `rag_top_similarity` p50 | резко упал | эмбеддер/коллекция рассогласовались |

Главный враг алертов — **усталость** (alert fatigue): если алертов много и они
ложные, дежурный их отключит. Лучше три точных алерта, чем тридцать шумных.
Порог ставьте по факту (посмотрите реальное распределение неделю), а не «на глаз».


## ⚠️ Частые ошибки

- **Логировать f-строкой.** `log.info(f"...{x}...")` — это текст, по нему не
  построишь запрос. Логируйте события с полями (`log_event("ask.done", ms=..)`).
- **ПДн/секреты/промпты в логах.** Логи живут долго и доступны многим. Через
  `scrub_for_logs`, а лучше — только длину и хеш вопроса.
- **Высокая кардинальность меток.** `user_id`, `trace_id`, текст вопроса в
  метке → взрыв временных рядов. Такие вещи — в логи (по `trace_id`), не в метки.
- **Смотреть на среднее.** Холодный старт и хвосты среднее прячет. Только
  перцентили: p50 — типичный запрос, p95/p99 — боль пользователей.
- **Алертить по логам грепом.** Логи — для разбора инцидента, алерты — по
  метрикам. Grep по ERROR не даст «доля отказов выросла до 40%».
- **Push вместо pull.** Prometheus сам приходит на `/metrics`. Не надо ничего
  «отправлять» (push-шлюз — исключение для батч-джоб, не для сервиса).


## Упражнения

1. **Дашборд в текст.** Напишите функцию, которая по тексту `/metrics` печатает
   мини-сводку: всего запросов, разбивка по `outcome` в процентах, сколько ПДн
   замаскировано, сколько инъекций отбито. Это ваш «дашборд для бедных» без Grafana.

2. **Алерт на свежесть индекса.** Добавьте в `check_alerts` правило: если
   `rag_index_last_success_timestamp_seconds` старше суток — алерт «индекс не
   обновлялся». Подумайте, откуда взять «сейчас» (подсказка: `time.time()`), и
   почему такой алерт спас бы нас от антипаттерна №1 из урока 4.1.

3. **Correlation по trace_id.** Включите JSON-логи, сделайте один запрос,
   соберите все строки с одним `trace_id` и восстановите по ним полный путь
   запроса (стадии и их длительности). Это ровно то, что вы будете делать при
   разборе продового инцидента.


## Мини-квиз

<details><summary><b>1. Метрика, лог и трейс — на какой вопрос отвечает каждый?</b></summary>

**Метрика** — «плохо ли *сейчас*, в целом по потоку?» (p95 латентности, доля
отказов). **Лог** — «что именно случилось в *конкретном* запросе X?» (детально,
по `trace_id`). **Трейс** — «на какой *стадии* этого запроса ушло время?»
(retrieve 20 мс, llm 7 с). Путать их — типичная ошибка: алертить по логам или
хранить в логах то, что должно быть метрикой.
</details>

<details><summary><b>2. Почему нельзя класть user_id в метку Prometheus-метрики?</b></summary>

Каждое уникальное значение метки создаёт отдельный временной ряд. `user_id`
(тысячи значений), `trace_id` или текст вопроса (бесконечность значений)
порождают взрыв рядов — **cardinality explosion**, который кладёт Prometheus по
памяти. Идентификаторы запроса — в логи (связываются по `trace_id`), в метки —
только поля с малым числом значений (`outcome`, `tenant`, `stage`).
</details>

<details><summary><b>3. Почему для латентности смотрят p95/p99, а не среднее?</b></summary>

Среднее прячет хвосты и холодные старты. Если 95% запросов — 2 секунды, а 5% —
40 секунд (холодный старт, очередь), среднее (~4 с) не описывает ни тех, ни
других, и «боль» 5% пользователей не видна. p95/p99 показывают именно этот хвост
— то, из-за чего приходят жалобы.
</details>

<details><summary><b>4. Prometheus работает по pull или push? Что это значит для сервиса?</b></summary>

По **pull**: сервис ничего не отправляет — он лишь отдаёт текущие значения
текстом на `GET /metrics`, а сервер Prometheus сам приходит за ними раз в
`scrape_interval`. Значит, сервису достаточно объявить метрики и открыть один
эндпоинт; никакой сетевой доставки, очередей и «а вдруг коллектор упал» — это
забота Prometheus.
</details>

<details><summary><b>5. Что логировать вместо текста вопроса пользователя и почему?</b></summary>

Длину (`question_len`) и хеш (`sha256[:12]`). Текст вопроса может содержать ПДн,
а логи хранятся долго и доступны многим. По хешу можно сгруппировать одинаковые
вопросы (например, найти самый частый) и связать события, не сохранив ни одного
реального вопроса. Если текст всё же нужен для разбора — только через
`scrub_for_logs` (маскирует ПДн и секреты).
</details>


## Решения

Три упражнения — три инструмента дежурного: сводка, будильник и лупа для
разбора инцидента. Все работают поверх текста `/metrics` и структурных логов,
которые сервис уже отдаёт.

**Решение 1 — дашборд в текст.** Grafana под рукой не всегда, а понять «как
себя чувствует сервис» нужно за секунды. Парсим текст `/metrics` и печатаем
сводку. Разбор `outcome` по процентам — самое полезное: скачок доли
`refused_no_context` — сигнал, что сломался индекс или порог.

In [ ]:
import re


def read_counter(metrics_text, name, **labels):
    """Сумма счётчика с нужными метками из текста /metrics (как в разделе про алерты)."""
    total = 0.0
    for line in metrics_text.splitlines():
        if not line.startswith(name) or line.startswith("#"):
            continue
        if all(f'{k}="{v}"' in line for k, v in labels.items()):
            total += float(line.rsplit(" ", 1)[-1])
    return total


def outcomes_in(metrics_text):
    found = {}
    for line in metrics_text.splitlines():
        if line.startswith("rag_requests_total") and "outcome=" in line and not line.startswith("#"):
            m = re.search(r'outcome="([^"]+)"', line)
            if m:
                found[m.group(1)] = found.get(m.group(1), 0.0) + float(line.rsplit(" ", 1)[-1])
    return found


def dashboard(metrics_text):
    total = read_counter(metrics_text, "rag_requests_total")
    print(f"всего запросов: {int(total)}")
    if total:
        for outcome, n in sorted(outcomes_in(metrics_text).items(), key=lambda kv: -kv[1]):
            print(f"  {outcome:22s} {int(n):>4}  ({n / total:4.0%})")
    pii = read_counter(metrics_text, "rag_pii_masked_total")
    inj = read_counter(metrics_text, "rag_injection_blocked_total")
    print(f"замаскировано ПДн: {int(pii)} | отбито инъекций: {int(inj)}")


# берём живой /metrics, если сервис уже опрашивали выше; иначе — образец
try:
    sample_metrics = metrics_text            # из раздела «Метрики в FastAPI»
except NameError:
    sample_metrics = (
        'rag_requests_total{outcome="answered",tenant="vectorika"} 7.0\n'
        'rag_requests_total{outcome="refused_no_context",tenant="vectorika"} 2.0\n'
        'rag_requests_total{outcome="blocked_output",tenant="vectorika"} 1.0\n'
        'rag_pii_masked_total{entity="EMAIL"} 3.0\n'
        'rag_pii_masked_total{entity="PHONE_RU"} 4.0\n'
        'rag_injection_blocked_total{stage="ingest",rule="credential_bait"} 2.0\n')

dashboard(sample_metrics)

Тот же `read_counter`, что питал алерты, годится и для сводки — метрики
переиспользуются. В проде такую функцию вешают на `/status` (для людей) рядом
с `/metrics` (для Prometheus).

**Решение 2 — алерт на свежесть индекса.** `rag_index_last_success_timestamp_seconds`
— это Gauge, который индексация выставляет в конце (`RAG_INDEX_LAST_SUCCESS.set(...)`).
«Сейчас» берём из `time.time()`, и если метка старше суток — бьём тревогу.
Такой алерт — прямая страховка от антипаттерна №1 из урока 4.1: устаревший
индекс отвечает «правдоподобно и неверно», исключений не бросает, и заметить
это можно только по возрасту последней успешной переиндексации.

In [ ]:
def check_freshness(metrics_text, now, max_age_s=86400):
    """Алерт свежести индекса. now — Unix-время (time.time()); порог — сутки."""
    ts = read_counter(metrics_text, "rag_index_last_success_timestamp_seconds")
    if ts == 0:
        return "⚠️ индекс ни разу не обновлялся (метрика = 0)"
    age = now - ts
    if age > max_age_s:
        return f"⚠️ индекс не обновлялся {age / 3600:.1f} ч (порог {max_age_s / 3600:.0f} ч)"
    return f"✓ индекс свежий (обновлён {age / 3600:.1f} ч назад)"


# «сейчас» фиксируем числом, чтобы прогон урока был воспроизводим;
# в проде здесь стоит time.time()
NOW = 1_700_000_000
fresh = f"rag_index_last_success_timestamp_seconds {NOW - 3600}\n"       # час назад
stale = f"rag_index_last_success_timestamp_seconds {NOW - 2 * 86400}\n"  # двое суток
print("свежий индекс:   ", check_freshness(fresh, NOW))
print("устаревший индекс:", check_freshness(stale, NOW))

В PromQL это правило пишется одной строкой:
`time() - rag_index_last_success_timestamp_seconds > 86400`. Наш Python —
та же мысль без Prometheus-сервера. Обратите внимание на ветку `ts == 0`: метрика
Gauge стартует с нуля, и «ни разу не обновлялся» — это тоже алерт, а не «всё ок».

**Решение 3 — correlation по trace_id.** Это ровно то, что делают при разборе
продового инцидента: включают JSON-логи, берут один `trace_id` и собирают по
нему весь путь запроса. Включаем JSON-формат, делаем один запрос, ловим его
логи и восстанавливаем стадии с длительностями:

In [ ]:
import io
import json as _json
from contextlib import redirect_stdout
from rag_service.observability import configure_logging, new_trace_id, stage

buf = io.StringIO()
with redirect_stdout(buf):
    configure_logging("INFO", json_logs=True)          # прод-формат: строка = JSON
    if QDRANT_UP and OLLAMA_UP:
        from rag_service.pipeline import RagPipeline
        from rag_service.access import DEMO_PRINCIPALS
        from rag_service.embeddings import get_embedder
        pipe = RagPipeline(kb=kb, embedder=get_embedder(), llm=OllamaLLM())
        pipe.build_lexical_index()
        result = pipe.ask("За сколько дней подавать заявку на отпуск?",
                          DEMO_PRINCIPALS["demo-employee-token"])
        target_id = result.trace_id
    else:
        # инфраструктуры нет — воспроизведём путь теми же примитивами стадий
        import time
        target_id = new_trace_id()
        for name, pause in [("guard.query", 0.001), ("retrieve.dense", 0.006),
                            ("retrieve.bm25", 0.001), ("llm.generate", 0.02)]:
            with stage(name):
                time.sleep(pause)
configure_logging("WARNING", json_logs=False)          # вернуть человекочитаемые логи

events = [_json.loads(l) for l in buf.getvalue().splitlines() if l.strip().startswith("{")]
mine = [e for e in events if e.get("trace_id") == target_id]
print(f"trace_id={target_id}: событий в логах — {len(mine)}\n")
print(f"{'стадия':24s}{'мс':>9}")
print("-" * 33)
total_ms = 0.0
for e in mine:
    if e.get("event") == "stage.done":
        total_ms += e.get("duration_ms", 0.0)
        print(f"{e['stage']:24s}{e.get('duration_ms', 0):>9}")
print("-" * 33)
print(f"{'ИТОГО по стадиям':24s}{total_ms:>9.1f}")

Строки с одним `trace_id` сложились в полный трейс запроса — и сразу
видно, **где** ушло время (почти всё — в `llm.generate`, поиск занял единицы
миллисекунд; ровно как обещал урок). В Kibana/Loki это один фильтр
`trace_id: "…"`, а `trace_id` возвращается ещё и в заголовке `X-Trace-Id`, так
что от жалобы пользователя («запрос abc123 завис») до полного трейса —
один запрос к логам. В этом весь смысл сквозного идентификатора из урока 3.7:
не протаскивать его руками, а положить в `contextvars` один раз на входе.

## Итоги

- **Три столпа наблюдаемости** — не синонимы: логи (что в запросе X), метрики
  (плохо ли сейчас в целом), трейсы (где стадия тормозит). Всё сшивается
  `trace_id`.
- Логи — **структурные** (JSON, поля, не f-строки), **без ПДн/секретов/промптов**;
  вместо текста вопроса — длина и хеш.
- Метрики Prometheus — Counter/Gauge/Histogram, **низкая кардинальность меток**,
  модель **pull** через `GET /metrics`.
- **Среднее лжёт**: холодный старт LLM (~37 с в нашем прогоне против ~2 с тёплых)
  прячется в среднем и виден только в p95/p99.
- **Алерты** — правила по метрикам (PromQL), точные и немногочисленные; лучше
  три честных, чем тридцать шумных.

## Что дальше

Мы видим сервис насквозь и знаем, что узкое место — поиск, когда он промахивается
(урок 4.1) и генерация, когда она медленная. Следующий урок берётся за качество
поиска вплотную: **4.4 «Оптимизация поиска: reranking и multi-query»** —
cross-encoder, слияние RRF и переформулировка запросов через LLM. Там мы наконец
починим тот самый семантический разрыв с «работой из Грузии».

## Полезные ссылки
- Prometheus — типы метрик: <https://prometheus.io/docs/concepts/metric_types/>
- Prometheus — histograms и quantiles: <https://prometheus.io/docs/practices/histograms/>
- prometheus-fastapi-instrumentator: <https://github.com/trallnag/prometheus-fastapi-instrumentator>
- Три столпа observability: <https://www.oreilly.com/library/view/distributed-systems-observability/9781492033431/>
- structlog (продвинутые структурные логи): <https://www.structlog.org>
